# Download Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/download_data.ipynb)

`quantem.data` provides small, public electron microscopy datasets hosted on Hugging Face. The pattern is: inspect the hosted files, download the dataset once into the local Hugging Face cache, then pass a NumPy array or quantem `Dataset2d` / `Dataset3d` object directly into the widgets.

This tutorial uses a real HAADF image of gold nanoparticles. The source array is 4096 by 4096 `uint16` data, so the notebook uses a calibrated preview crop for the interactive documentation page while keeping the full cached file available locally.

In [1]:
from huggingface_hub import hf_hub_download
import numpy as np
import quantem.data as qdata

from quantem.core.datastructures import Dataset2d, Dataset3d
from quantem.widget import Show2D, Show3D

## Find a dataset

Use `quantem.data.list_files()` for quick discovery. The table below is intentionally filtered to small HAADF files so the documentation page does not trigger a large 4D-STEM download.

In [2]:
haadf_files = [
    item for item in qdata.list_files("haadf")
    if item["type"] == "data"
]
haadf_files

[{'path': 'haadf/gold_drift_0deg.emd', 'size_mb': 8.32, 'type': 'data'},
 {'path': 'haadf/gold_drift_90deg.emd', 'size_mb': 8.32, 'type': 'data'},
 {'path': 'haadf/gold_haadf.emd', 'size_mb': 32.28, 'type': 'data'},
 {'path': 'haadf/gold_haadf_npy/data.npy', 'size_mb': 32.0, 'type': 'data'}]

## Load a real HAADF image

The current public registry stores this NPY dataset under a nested Hugging Face path. We download that exact file and open it with `mmap_mode="r"`, so reading metadata is immediate and the full 32 MB array is not copied until the preview is sliced.

In [3]:
repo_id = "bobleesj/quantem-data"
filename = "haadf/gold_haadf_npy/data.npy"
local_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")

full_image = np.load(local_path, mmap_mode="r")
{
    "shape": full_image.shape,
    "dtype": str(full_image.dtype),
    "hosted_file": filename,
}

{'shape': (4096, 4096),
 'dtype': 'uint16',
 'hosted_file': 'haadf/gold_haadf_npy/data.npy'}

## Show2D preview

For docs and Colab startup speed, take a 512 by 512 preview from the full image. The sampling is scaled by the stride so the scale bar remains calibrated.

In [4]:
stride = 8
preview = np.asarray(full_image[::stride, ::stride], dtype=np.float32)
preview -= preview.min()
preview /= preview.max() or 1.0

pixel_size_nm = 0.018 * stride
dataset2d = Dataset2d.from_array(
    preview,
    sampling=(pixel_size_nm, pixel_size_nm),
    units=("nm", "nm"),
    name="Gold HAADF preview",
)

Show2D(dataset2d, offline=True, cmap="inferno")

Show2D(512×512, cmap=inferno)

## Show3D from downloaded data

A time or depth stack can use the same download/cache pattern. For a lightweight demo, this cell makes a small aligned stack from the downloaded HAADF preview so `Show3D` has real image texture without downloading another large file.

In [5]:
shifts = np.linspace(-12, 12, 24).astype(int)
stack = np.stack([np.roll(preview, shift, axis=0) for shift in shifts]).astype(np.float32)
stack_dataset = Dataset3d.from_array(
    stack,
    sampling=(1.0, pixel_size_nm, pixel_size_nm),
    units=("frame", "nm", "nm"),
    name="Gold HAADF preview stack",
)

Show3D(stack_dataset, offline=True, fps=12, cmap="inferno")

Show3D(24×512×512, frame=12, cmap=inferno)

## Larger files

Large raw or 4D-STEM files are available through the same Hugging Face cache, but keep those downloads out of documentation execution unless the page is specifically about that dataset.

```python
raw_files = qdata.list_files("4dstem")
# Pick a raw file intentionally, then download it with huggingface_hub.hf_hub_download
# or the matching quantem.data raw helper once the dataset is registered there.
```

For shareable examples, use a small calibrated preview in the notebook and link to the full dataset path. For local analysis, load the full cached file and pass the full array into the widget.